In [18]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

# Use a BASE model instead of an Instruct model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./outputs/sft_full_dataset/checkpoint-500",
    # model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False, # <--- SET THIS TO FALSE
)

# Ensure pad token is set for base models (often required for batched generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=True,
    target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ]
)


==((====))==  Unsloth 2026.4.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.19.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Already have LoRA adapters! We shall skip this step.


In [19]:

# ==========================================
# 2. DATASET PREPARATION
# ==========================================
import re

def format_dataset(row):
    # RELIABLE EXTRACTION: Parse directly from the JSON column, not the prompt string.
    # try:
    examples = row['examples']
    input_tests = [str(ex['input']) for ex in examples]
    output_tests = [str(ex['output']) for ex in examples]
    # except Exception:
    #     input_tests = []
    #     output_tests = []

    # BASE MODEL PROMPT: Force the model into the right format.
    raw_prompt = (
        "You compulsorily have to think and put your reasoning inside <think> </think> tags, and your code inside a markdown block.\n\n"
        f"{row['prompt']}\n\n"
        "<think>\n"
    )

    return {
        "prompt": raw_prompt,
        "input_tests": input_tests,
        "output_tests": output_tests,
    }

print("Loading and filtering dataset...")
# Load dataset and filter for 'A' questions
dataset = load_dataset("open-r1/codeforces-cots", split="train[:1000]")

filtered_dataset = dataset.filter(lambda x: x['id'].endswith('A'))
processed_dataset = filtered_dataset.map(format_dataset, num_proc=4)

train_dataset = processed_dataset.select(range(min(1000, len(processed_dataset))))


Loading and filtering dataset...


In [20]:
from pprint import pprint
pprint(train_dataset[0]['prompt'])
pprint(train_dataset[0]['input_tests'])
pprint(train_dataset[0]['output_tests'])

('You compulsorily have to think and put your reasoning inside <think> '
 '</think> tags, and your code inside a markdown block.\n'
 '\n'
 'You will be given a competitive programming problem. Please reason step by '
 'step about the solution, then provide a complete implementation in C++17.\n'
 '\n'
 'Your solution must read input from standard input (cin), write output to '
 'standard output (cout).\n'
 'Do not include any debug prints or additional output.\n'
 '\n'
 'Put your final solution within a single code block:\n'
 '```cpp\n'
 '<your code here>\n'
 '```\n'
 '\n'
 '# Problem\n'
 '\n'
 'You are given an array $$$a$$$ of $$$n$$$ integers, where $$$n$$$ is odd.\n'
 '\n'
 'In one operation, you will remove two adjacent elements from the array '
 '$$$a$$$, and then concatenate the remaining parts of the array. For example, '
 'given the array $$$[4,7,4,2,9]$$$, we can obtain the arrays $$$[4,2,9]$$$ '
 'and $$$[4,7,9]$$$ by the operations $$$[\\underline{4,7}, 4,2,9] \\to '
 '[4,2,

In [21]:
import re
import os
import sys
import subprocess
import tempfile

# ==========================================
# 2. REWARD FUNCTIONS
# ==========================================
def format_reward_func(completions, **kwargs):
    rewards = []
    for c in completions:
        content = c[0]['content'] if isinstance(c, list) else c

        has_think_close = bool(re.search(r"</think>", content))
        has_code = bool(re.search(r"```(python|cpp)\n.*?```", content, re.DOTALL))

        rewards.append(0.2 if (has_think_close and has_code) else 0.0)
    return rewards

def extract_code(completion):
    match = re.search(r"```(python|cpp)\n(.*?)```", completion, re.DOTALL)
    if match:
        return match.group(1), match.group(2)
    return None, None

def correctness_reward_func(completions, input_tests, output_tests, **kwargs):
    rewards = []

    for comp, ins, outs in zip(completions, input_tests, output_tests):
        content = comp[0]['content'] if isinstance(comp, list) else comp
        lang, code = extract_code(content)

        # Fail early if there is no code, or if the dataset parser failed
        if not code or not ins or len(ins) == 0:
            rewards.append(-1.0) # Standard negative reward for failing format/parsing
            continue

        total = len(ins)
        score = 0
        compilation_failed = False

        # Compile ONCE per completion, run ALL test cases using that executable
        with tempfile.TemporaryDirectory() as tmp:
            if lang == "cpp":
                cpp = os.path.join(tmp, "sol.cpp")
                exe = os.path.join(tmp, "sol")
                with open(cpp, "w") as f: f.write(code)

                # Compile Phase
                compile_res = subprocess.run(["g++", "-O2", cpp, "-o", exe], capture_output=True)
                if compile_res.returncode != 0:
                    compilation_failed = True
                cmd = [exe]
            else:
                py = os.path.join(tmp, "sol.py")
                with open(py, "w") as f: f.write(code)
                cmd = [sys.executable, py]

            # Execution Phase (only if compilation succeeds)
            if not compilation_failed:
                for i, o in zip(ins, outs):
                    try:
                        res = subprocess.run(cmd, input=i, text=True, capture_output=True, timeout=2.0)
                        if res.returncode == 0 and res.stdout.strip() == o.strip():
                            score += 1
                    except subprocess.TimeoutExpired:
                        pass # Treat TLE as a failed test case
                    except Exception:
                        pass # Treat other runtime errors as a failed test case

        # ----------------------------------------------------
        # NEW REWARD LOGIC BASED ON YOUR RULES
        # ----------------------------------------------------
        if compilation_failed:
            rewards.append(-2.0)  # Negative High Reward
            print("Compilation failed.")

        elif score == total:
            rewards.append(2.0)   # High Positive Reward
            print(f"All test cases passed! ({score}/{total})")

        elif score > total / 2:
            # Positive Partial Reward: Base of 0.5 + scaled bonus for passing rate
            partial_reward = 0.5 + 0.5 * (score / total)
            rewards.append(partial_reward)
            print(f"Passed majority: {score}/{total} test cases.")

        else:
            # Less than or equal to 50% passed
            rewards.append(-0.5)  # Negative Small Reward
            print(f"Passed minority: {score}/{total} test cases.")

    return rewards

In [22]:
temp_prompt = train_dataset[0]['prompt']

# 2. Tokenize the raw string directly (no chat template needed here)
inputs = tokenizer(
    temp_prompt,
    return_tensors="pt"
).to("cuda")

# 3. Generate the response
outputs = model.generate(
    **inputs,
    max_new_tokens=1024,
    pad_token_id=tokenizer.eos_token_id,
    use_cache=True
)

# 4. Decode ONLY the newly generated tokens (slice off the input prompt length)
generated_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(generated_text)


Okay, let's see. The problem is about removing pairs of adjacent elements from an array until only one remains. We need to find the maximum possible value of that remaining element.

Hmm. So the key is to figure out which elements can be left at the end, given that each operation removes two adjacent elements. The order in which we remove pairs matters a lot because each removal affects the positions of the remaining elements.

Wait, but since each operation removes two adjacent elements, the remaining elements form a subsequence of the original array, but with certain constraints. For example, each element after the first must be adjacent in the original array. Or maybe not exactly adjacent, but the way the removals happen.

Wait, let's think about how the elements are removed. Suppose we have an array like [a, b, c, d, e]. If we remove b and c, then the array becomes [a, d, e]. Then if we remove d and e, we get [a]. So the remaining element is a. Alternatively, if we remove a and b 

In [23]:
# 1. Reconstruct the full text so the regex format checker works properly
full_completion = "<think>\n" + generated_text


# 2. Extract the test cases for this specific problem (index 0)
# Note: The reward functions expect a nested list (a batch containing one list of tests)
input_tests = [processed_dataset[0]["input_tests"]]
output_tests = [processed_dataset[0]["output_tests"]]

print("\n--- Running Evaluation ---")
# 3. Run the Format Reward
fmt_score = format_reward_func([full_completion])[0]
print(f"Format Score      : {fmt_score}")

# 4. Run the Correctness Reward (Compiles and runs the code against the tests)
corr_score = correctness_reward_func([full_completion], input_tests, output_tests)[0]
print(f"Correctness Score : {corr_score}")

# Optional: Print the extracted code to see what was actually compiled
lang, extracted_code = extract_code(full_completion)
if extracted_code:
    print(f"\n--- Extracted {lang.upper()} Code ---")
    print(extracted_code)
else:
    print("\n--- Failed to extract valid code block ---")


--- Running Evaluation ---
Format Score      : 0.0
Correctness Score : -1.0

--- Failed to extract valid code block ---


In [ ]:
csc = 0
total = 0
for i in range(-10,0):
  # ==========================================
  # 5. POST-TRAINING VERIFICATION
  # ==========================================
  print("\n--- Running Manual Verification ---")
  val_sample = processed_dataset[i]

  # BASE MODEL CHANGE: Use direct text tokenization instead of apply_chat_template
  prompt_text = val_sample["prompt"]
  inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

  print(f"Generating solution for Problem: {val_sample['id']}")
  outputs = model.generate(
      **inputs,
      max_new_tokens=1024,
      pad_token_id=tokenizer.pad_token_id
  )

  # Decode only the generated portion
  generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

  print("\n--- Generated Output ---")
  print(generated_text)
  print("------------------------")

  format_score = format_reward_func([generated_text])[0]
  correctness_score = correctness_reward_func(
      [generated_text],
      [val_sample["input_tests"]],
      [val_sample["output_tests"]]
  )[0]

  print("\n--- Metrics ---")
  print(f"Format Reward  : {format_score}")
  print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")

  csc+=correctness_score
  total+=1
print(csc/total)


--- Running Manual Verification ---
Generating solution for Problem: 928/A



--- Generated Output ---

Okay, let's see. The problem is to check if a new user login is similar to any existing ones. The similarity is defined by certain transformations. So, the key is to normalize the login in a way that all similar logins become the same, and then check if the normalized new login exists in the normalized existing ones.

Hmm, right. So the approach here is to create a canonical form for each login. The canonical form should be the login after applying all possible transformations allowed by the problem. Then, if the new login's canonical form is already present in the existing ones' canonical forms, output No, else Yes.

So how do we generate this canonical form?

Let's think about the transformations allowed:

1. Case insensitivity. So all letters are considered the same regardless of uppercase or lowercase. So we can convert all characters to lowercase (or uppercase) first.

Wait, but the problem says that transforming lowercase to uppercase and vice versa is 

In [ ]:
# 1. UNSLOTH MUST BE IMPORTED FIRST
from unsloth import FastLanguageModel

# 2. Then import everything else
import os
import re
import sys
import json
import tempfile
import subprocess
from datasets import load_dataset
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer

max_seq_length = 2048

# Use a BASE model instead of an Instruct model
model, tokenizer = FastLanguageModel.from_pretrained(
    # model_name="unsloth/Qwen2.5-0.5B",
    model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True, # <--- SET THIS TO FALSE
)

# Ensure pad token is set for base models (often required for batched generation)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)



csc = 0
total = 0
for i in range(-10,0):
  # ==========================================
  # 5. POST-TRAINING VERIFICATION
  # ==========================================
  print("\n--- Running Manual Verification ---")
  val_sample = processed_dataset[i]

  # BASE MODEL CHANGE: Use direct text tokenization instead of apply_chat_template
  prompt_text = val_sample["prompt"]
  inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

  print(f"Generating solution for Problem: {val_sample['id']}")
  outputs = model.generate(
      **inputs,
      max_new_tokens=10000,
      pad_token_id=tokenizer.pad_token_id
  )

  # Decode only the generated portion
  generated_text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

  print("\n--- Generated Output ---")
  print(generated_text)
  print("------------------------")

  format_score = format_reward_func([generated_text])[0]
  correctness_score = correctness_reward_func(
      [generated_text],
      [val_sample["input_tests"]],
      [val_sample["output_tests"]]
  )[0]

  print("\n--- Metrics ---")
  print(f"Format Reward  : {format_score}")
  print(f"Correctness    : {correctness_score} (Percentage of Test Cases Passed)")

  csc+=correctness_score
  total+=1
print(csc/total)
